# LeetCode 2859 — Sum of Values at Indices With K Set Bits bằng Python

Link đề: https://leetcode.com/problems/sum-of-values-at-indices-with-k-set-bits/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ta không đếm bit của **giá trị** `nums[i]` mà đếm bit `1` trong **chỉ số** `i` (viết ở hệ nhị phân), rồi cộng các `nums[i]` thỏa điều kiện.
2. Biết **set bit** là gì, biết đổi số sang nhị phân bằng `bin()`, và đếm bit `1` bằng nhiều cách: `bin(i).count("1")`, dịch bit `>>` kèm `& 1`, thuật toán Kernighan `x & (x - 1)`, `int.bit_count()` và bảng quy hoạch động `pc[i] = pc[i >> 1] + (i & 1)`.
3. Đi từ cách **đổi sang chuỗi nhị phân rồi đếm** (dễ nghĩ nhất) → **dịch bit** → **Kernighan** → **tiền xử lý bảng popcount** → **hàm dựng sẵn `bit_count`**.
4. Biết tự **tính tay** một ví dụ, viết **trace** từng chỉ số, và **kiểm tra chéo** nhiều cách trên dữ liệu ngẫu nhiên có seed.
5. Tránh các lỗi kinh điển: đếm bit của giá trị thay vì chỉ số, nhầm chỉ số 1-indexed, quên `k = 0` (chỉ số `0` có đúng 0 bit), nhầm `&` với `and`.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên **đánh chỉ số từ 0** `nums` và một số nguyên `k`.

Hãy trả về **tổng các phần tử `nums[i]`** mà chỉ số `i` có **đúng `k` set bit** khi viết `i` ở dạng nhị phân.

> **Set bit** là một bit có giá trị `1` trong biểu diễn nhị phân.

Ví dụ: số `21` viết nhị phân là `10101` nên có **3 set bit**.

**Ví dụ 1**

- Input: `nums = [5, 10, 1, 5, 2]`, `k = 1`
- Output: `13`
- Giải thích: chỉ số `0..4` viết nhị phân là `000, 001, 010, 011, 100`. Các chỉ số có đúng 1 set bit là `1, 2, 4`. Tổng `nums[1] + nums[2] + nums[4] = 10 + 1 + 2 = 13`.

**Ví dụ 2**

- Input: `nums = [4, 3, 2, 1]`, `k = 2`
- Output: `1`
- Giải thích: chỉ số `0..3` là `00, 01, 10, 11`. Chỉ có chỉ số `3` (`11`) có đúng 2 set bit nên đáp án là `nums[3] = 1`.

**Constraints**

- `1 <= nums.length <= 1000`
- `1 <= nums[i] <= 10^5`
- `0 <= k <= 10`

Cell dưới đây chạy thử "bằng mắt" ví dụ 1: in chỉ số, dạng nhị phân và số bit `1` của từng chỉ số.

In [1]:
nums = [5, 10, 1, 5, 2]
k = 1
for i in range(len(nums)):
    print(i, format(i, "03b"), "->", bin(i).count("1"), "bit 1")
# Chỉ số 1, 2, 4 có đúng 1 bit 1 -> 10 + 1 + 2 = 13
print("Đáp án mong đợi:", 10 + 1 + 2)

0 000 -> 0 bit 1
1 001 -> 1 bit 1
2 010 -> 1 bit 1
3 011 -> 2 bit 1
4 100 -> 1 bit 1
Đáp án mong đợi: 13


## 3. Kiến thức nền

Trước khi giải, ta làm quen các công cụ Python sẽ dùng. Mỗi công cụ đi kèm một ví dụ nhỏ.

### 3.1. Hệ nhị phân và "set bit"

Máy tính lưu số nguyên bằng các bit `0` / `1`. Mỗi bit đứng ở một vị trí có "trọng số" là lũy thừa của 2:

`13 = 1101₂ = 1·8 + 1·4 + 0·2 + 1·1`.

Vậy `13` có **3 set bit** (ba chữ số `1`).

### 3.2. `bin(x)` — đổi số sang chuỗi nhị phân

`bin(x)` trả về chuỗi bắt đầu bằng tiền tố `0b`. Muốn bỏ tiền tố ta cắt `[2:]`, hoặc dùng `format(x, "b")`. Muốn thêm số `0` đứng đầu cho đủ độ rộng, dùng `format(x, "08b")`.

In [2]:
x = 13
print(bin(x))                 # 0b1101
print(bin(x)[2:])             # 1101
print(format(x, "b"))         # 1101
print(format(x, "08b"))       # 00001101  (đủ 8 ký tự)
print(bin(0))                 # 0b0  -> số 0 không có bit 1 nào

0b1101
1101
1101
00001101
0b0


### 3.3. Đếm ký tự trong chuỗi: `str.count`

`"1101".count("1")` đếm số lần ký tự `"1"` xuất hiện. Ghép với `bin`, ta được cách đếm set bit ngắn nhất.

In [3]:
s = "1101"
print(s.count("1"))           # 3
print(bin(13).count("1"))     # 3  (tiền tố "0b" không chứa ký tự "1" nên không làm sai kết quả)
print(bin(1023).count("1"))   # 10

3
3
10


### 3.4. Phép AND bit `&` và phép dịch `>>`

- `x & 1` lấy **bit cuối** của `x` (`1` nếu `x` lẻ, `0` nếu `x` chẵn).
- `x >> 1` **dịch phải 1 bit**, tương đương `x // 2`: bỏ bit cuối.

Lặp "lấy bit cuối rồi bỏ nó" sẽ duyệt hết các bit của `x` từ phải sang trái. Đây giống hệt việc `% 10` và `// 10` để duyệt chữ số thập phân, nhưng cơ số là 2.

In [4]:
x = 13                        # 1101
print(x & 1, x >> 1, bin(x >> 1))     # 1 6 0b110
x >>= 1
print(x & 1, x >> 1, bin(x >> 1))     # 0 3 0b11
x >>= 1
print(x & 1, x >> 1, bin(x >> 1))     # 1 1 0b1
x >>= 1
print(x & 1, x >> 1, bin(x >> 1))     # 1 0 0b0

1 6 0b110
0 3 0b11
1 1 0b1
1 0 0b0


### 3.5. `x & (x - 1)` — xóa bit `1` thấp nhất

Khi trừ 1 khỏi `x`, bit `1` thấp nhất bị lật thành `0` và mọi bit `0` bên phải nó thành `1`. Do đó `x & (x - 1)` **xóa đúng một bit `1` (bit `1` thấp nhất)**.

Ví dụ `x = 12 = 1100₂`: `x - 1 = 1011₂`, `x & (x - 1) = 1000₂ = 8`.

In [5]:
x = 12
print(format(x, "04b"), format(x - 1, "04b"), format(x & (x - 1), "04b"))   # 1100 1011 1000
x = x & (x - 1)
print(x)                      # 8
x = x & (x - 1)
print(x)                      # 0  -> đã xóa hết 2 bit 1

1100 1011 1000
8
0


### 3.6. `int.bit_count()`

Từ Python 3.10, mọi số nguyên có phương thức `bit_count()` trả về số bit `1`. Nếu môi trường cũ hơn, dùng `bin(x).count("1")`.

In [6]:
import sys
print(sys.version_info[:2] >= (3, 10))
print((13).bit_count() if hasattr(int, "bit_count") else bin(13).count("1"))   # 3

True
3


### 3.7. `enumerate`, `sum` với generator

- `for i, v in enumerate(nums)` cho ta cả chỉ số `i` lẫn giá trị `v`.
- `sum(v for i, v in enumerate(nums) if điều_kiện)` cộng những `v` thỏa điều kiện (generator expression, không tạo list trung gian).

In [7]:
nums = [5, 10, 1, 5, 2]
for i, v in enumerate(nums):
    print("chỉ số", i, "giá trị", v)
print(sum(v for i, v in enumerate(nums) if i % 2 == 0))    # 5 + 1 + 2 = 8 (chỉ số chẵn)

chỉ số 0 giá trị 5
chỉ số 1 giá trị 10
chỉ số 2 giá trị 1
chỉ số 3 giá trị 5
chỉ số 4 giá trị 2
8


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Đề gồm hai việc tách biệt:

1. **Quyết định**: chỉ số `i` nào "được chọn"? Đó là chỉ số có đúng `k` bit `1`.
2. **Cộng**: cộng `nums[i]` của các chỉ số được chọn.

Tính tay ví dụ 2: `nums = [4, 3, 2, 1]`, `k = 2`.

| i | nhị phân | số bit 1 | bằng k = 2? | cộng `nums[i]` |
|---|---|---|---|---|
| 0 | 00 | 0 | không | — |
| 1 | 01 | 1 | không | — |
| 2 | 10 | 1 | không | — |
| 3 | 11 | 2 | **có** | `+1` |

Tổng = `1`. Nhận xét quan trọng: **giá trị** `nums[i]` (4, 3, 2, 1) hoàn toàn không ảnh hưởng đến việc có chọn `i` hay không; chỉ **chỉ số** quyết định.

## 5. Cách 1 — Brute Force / baseline: đổi từng chỉ số sang chuỗi nhị phân rồi đếm

Ý tưởng "thật thà" nhất: với mỗi `i`, đổi `i` thành chuỗi nhị phân, đếm ký tự `"1"`, nếu bằng `k` thì cộng `nums[i]`.

Ta tách riêng hàm `popcount_str(i)` để dễ so sánh với các cách sau.

In [8]:
from typing import List

def popcount_str(i: int) -> int:
    """Đếm bit 1 bằng chuỗi nhị phân."""
    return bin(i).count("1")

def sum_k_bits_bruteforce(nums: List[int], k: int) -> int:
    total = 0
    for i in range(len(nums)):
        if popcount_str(i) == k:
            total += nums[i]
    return total

print(sum_k_bits_bruteforce([5, 10, 1, 5, 2], 1))   # 13
print(sum_k_bits_bruteforce([4, 3, 2, 1], 2))       # 1

13
1


**Độ phức tạp:** gọi `n = len(nums)`. Mỗi chỉ số `i < n` có `O(log n)` bit, nên `bin(i)` và `count` tốn `O(log n)`. Tổng **Time `O(n log n)`**, **Space `O(log n)`** (chuỗi tạm). Với `n ≤ 1000` thì chỉ khoảng vài nghìn thao tác — hoàn toàn đủ nhanh.

**Nút thắt:** mỗi lần đếm bit ta tạo ra một chuỗi tạm. Không sai, nhưng nếu muốn "hiểu bit" thật sự hoặc làm bài phỏng vấn thì nên biết cách đếm bit **không cần chuỗi**.

## 6. Ý tưởng thuật toán / bước đột phá

Việc cần làm chỉ là **đếm popcount** (số bit 1) của `i`. Có ba hướng cải tiến, đều giữ nguyên khung "duyệt chỉ số, lọc, cộng":

1. **Dịch bit:** lặp `i & 1`, `i >>= 1` cho tới khi `i = 0`. Không tạo chuỗi.
2. **Kernighan:** lặp `i &= i - 1` và đếm số lần cho tới khi `i = 0`. Số vòng đúng bằng **số bit 1** (không phải số bit tổng).
3. **Bảng quy hoạch động:** `pc[i] = pc[i >> 1] + (i & 1)`. Vì `i >> 1` nhỏ hơn `i` nên đã tính xong; tính cả bảng popcount cho `0..n-1` trong `O(n)`.

Ngoài ra Python có sẵn `int.bit_count()` làm đúng việc đó bằng mã C rất nhanh.

Điểm chung: phần "cộng `nums[i]` nếu popcount bằng `k`" không đổi, chỉ có công cụ đếm bit thay đổi.

## 7. Các cách tiếp theo

### Cách 2 — Dịch bit từng bước (không dùng chuỗi)

**Giải thích từng đoạn.**

- `count = 0` lưu số bit 1 đã thấy.
- `while x:` lặp tới khi `x` bằng 0 (đã bỏ hết bit).
- `count += x & 1` cộng bit cuối (0 hoặc 1).
- `x >>= 1` bỏ bit cuối.

Ví dụ `x = 13 = 1101₂` đi qua các bit `1, 0, 1, 1`, tổng `3`.

In [9]:
def popcount_shift(x: int) -> int:
    count = 0
    while x:
        count += x & 1
        x >>= 1
    return count

print([popcount_shift(i) for i in range(8)])        # [0, 1, 1, 2, 1, 2, 2, 3]

def sum_k_bits_shift(nums: List[int], k: int) -> int:
    total = 0
    for i, v in enumerate(nums):
        if popcount_shift(i) == k:
            total += v
    return total

print(sum_k_bits_shift([5, 10, 1, 5, 2], 1))        # 13

[0, 1, 1, 2, 1, 2, 2, 3]
13


In [10]:
# Code nộp LeetCode cho Cách 2
class SolutionShift:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        total = 0
        for i, v in enumerate(nums):
            x = i
            count = 0
            while x:
                count += x & 1
                x >>= 1
            if count == k:
                total += v
        return total

print(SolutionShift().sumIndicesWithKSetBits([4, 3, 2, 1], 2))   # 1

1


**Độ phức tạp Cách 2:** mỗi chỉ số mất `O(log n)` vòng (số bit của `i`). Tổng **Time `O(n log n)`**, **Space `O(1)`**.

---

### Cách 3 — Thuật toán Kernighan (`x & (x - 1)`)

**Giải thích.** Mỗi lần `x &= x - 1` xóa đúng một bit 1 thấp nhất. Do đó số lần lặp để đưa `x` về `0` **bằng đúng số bit 1**. Với chỉ số ít bit 1 (như `1024`, `512`) vòng lặp rất ngắn.

In [11]:
def popcount_kernighan(x: int) -> int:
    count = 0
    while x:
        x &= x - 1      # xóa bit 1 thấp nhất
        count += 1
    return count

print([popcount_kernighan(i) for i in range(8)])    # [0, 1, 1, 2, 1, 2, 2, 3]
print(popcount_kernighan(1023), popcount_kernighan(1024))   # 10 1

[0, 1, 1, 2, 1, 2, 2, 3]
10 1


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionKernighan:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        total = 0
        for i, v in enumerate(nums):
            x, count = i, 0
            while x:
                x &= x - 1
                count += 1
            if count == k:
                total += v
        return total

print(SolutionKernighan().sumIndicesWithKSetBits([5, 10, 1, 5, 2], 1))   # 13

13


**Độ phức tạp Cách 3:** mỗi chỉ số tốn `O(số bit 1 của i)` ≤ `O(log n)`. Tổng **Time `O(n log n)` trong trường hợp xấu nhất**, **Space `O(1)`**. Trung bình nhanh hơn Cách 2 vì không duyệt các bit 0.

---

### Cách 4 — Bảng quy hoạch động popcount

**Giải thích.** Với `i ≥ 1`: bỏ bit cuối của `i` ta được `i >> 1` (nhỏ hơn `i`), nên

`popcount(i) = popcount(i >> 1) + (i & 1)`.

Ví dụ `popcount(13) = popcount(6) + 1 = 2 + 1 = 3`. Ta dựng bảng `pc` từ `0` tới `n - 1` rồi tra cứu. Cách này đáng giá khi cần popcount của **rất nhiều** số liên tiếp.

In [13]:
n = 8
pc = [0] * n
for i in range(1, n):
    pc[i] = pc[i >> 1] + (i & 1)
    print(f"pc[{i}] = pc[{i >> 1}] + {i & 1} = {pc[i]}")
print(pc)

pc[1] = pc[0] + 1 = 1
pc[2] = pc[1] + 0 = 1
pc[3] = pc[1] + 1 = 2
pc[4] = pc[2] + 0 = 1
pc[5] = pc[2] + 1 = 2
pc[6] = pc[3] + 0 = 2
pc[7] = pc[3] + 1 = 3
[0, 1, 1, 2, 1, 2, 2, 3]


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionTable:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        n = len(nums)
        pc = [0] * n
        for i in range(1, n):
            pc[i] = pc[i >> 1] + (i & 1)
        total = 0
        for i in range(n):
            if pc[i] == k:
                total += nums[i]
        return total

print(SolutionTable().sumIndicesWithKSetBits([4, 3, 2, 1], 2))   # 1

1


**Độ phức tạp Cách 4:** dựng bảng `O(n)`, duyệt `O(n)`. Tổng **Time `O(n)`**, **Space `O(n)`** cho bảng `pc`.

---

### Cách 5 — `int.bit_count()` (ngắn nhất)

**Giải thích.** `i.bit_count()` làm đúng việc đếm bit 1. Cả bài còn một dòng nhờ `sum` + generator. Nếu LeetCode/IDE của bạn dùng Python < 3.10, hãy dùng `bin(i).count("1")`.

In [15]:
class SolutionBitCount:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        return sum(v for i, v in enumerate(nums) if i.bit_count() == k)

print(SolutionBitCount().sumIndicesWithKSetBits([5, 10, 1, 5, 2], 1))   # 13

13


**Độ phức tạp Cách 5:** `bit_count()` chạy theo số "từ" (word) của số nguyên; với số nhỏ coi như `O(1)`. Tổng **Time `O(n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Trace Cách 3 (Kernighan) cho `nums = [5, 10, 1, 5, 2]`, `k = 1`:

| i | nhị phân | các lần `x &= x-1` | popcount | `== k`? | `nums[i]` | total sau |
|---|---|---|---|---|---|---|
| 0 | 000 | (không lặp) | 0 | không | 5 | 0 |
| 1 | 001 | 001→000 | 1 | **có** | 10 | 10 |
| 2 | 010 | 010→000 | 1 | **có** | 1 | 11 |
| 3 | 011 | 011→010→000 | 2 | không | 5 | 11 |
| 4 | 100 | 100→000 | 1 | **có** | 2 | 13 |

In [16]:
def trace_kernighan(nums, k):
    total = 0
    for i, v in enumerate(nums):
        x, count, steps = i, 0, [format(i, "03b")]
        while x:
            x &= x - 1
            count += 1
            steps.append(format(x, "03b"))
        chon = (count == k)
        if chon:
            total += v
        print(f"i={i} nhị phân={format(i, '03b')} chuỗi bước={'->'.join(steps)} "
              f"popcount={count} chọn={chon} nums[i]={v} total={total}")
    return total

print("kết quả:", trace_kernighan([5, 10, 1, 5, 2], 1))

i=0 nhị phân=000 chuỗi bước=000 popcount=0 chọn=False nums[i]=5 total=0
i=1 nhị phân=001 chuỗi bước=001->000 popcount=1 chọn=True nums[i]=10 total=10
i=2 nhị phân=010 chuỗi bước=010->000 popcount=1 chọn=True nums[i]=1 total=11
i=3 nhị phân=011 chuỗi bước=011->010->000 popcount=2 chọn=False nums[i]=5 total=11
i=4 nhị phân=100 chuỗi bước=100->000 popcount=1 chọn=True nums[i]=2 total=13
kết quả: 13


In [17]:
def trace_table(nums, k):
    n = len(nums)
    pc = [0] * n
    for i in range(1, n):
        pc[i] = pc[i >> 1] + (i & 1)
    print("bảng popcount pc =", pc)
    chosen = [i for i in range(n) if pc[i] == k]
    print("các chỉ số được chọn:", chosen)
    print("các giá trị tương ứng:", [nums[i] for i in chosen])
    return sum(nums[i] for i in chosen)

print("kết quả:", trace_table([5, 10, 1, 5, 2], 1))
print("kết quả:", trace_table([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 2))   # chỉ số 3, 5, 6, 9

bảng popcount pc = [0, 1, 1, 2, 1]
các chỉ số được chọn: [1, 2, 4]
các giá trị tương ứng: [10, 1, 2]
kết quả: 13
bảng popcount pc = [0, 1, 1, 2, 1, 2, 2, 3, 1, 2]
các chỉ số được chọn: [3, 5, 6, 9]
các giá trị tương ứng: [1, 1, 1, 1]
kết quả: 4


## 9. Code nộp LeetCode

Dùng cách đếm bằng dịch bit (đúng ở mọi phiên bản Python, không phụ thuộc `bit_count`). Phần import `List` là của môi trường LeetCode, ở notebook ta tự import.

In [18]:
from typing import List

class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        total = 0
        for i, v in enumerate(nums):
            x, count = i, 0
            while x:
                count += x & 1
                x >>= 1
            if count == k:
                total += v
        return total

print(Solution().sumIndicesWithKSetBits([5, 10, 1, 5, 2], 1))   # 13
print(Solution().sumIndicesWithKSetBits([4, 3, 2, 1], 2))       # 1

13
1


## 10. Vì sao thuật toán đúng

- **Không bỏ sót:** vòng `for` đi qua mọi chỉ số `0 … n-1`, mỗi chỉ số được kiểm tra điều kiện đúng một lần.
- **Không chọn nhầm:** `count` sau vòng `while` bằng số bit 1 của `i`. Mỗi bước `count += x & 1` cộng bit cuối rồi `x >>= 1` bỏ bit đó, nên sau khi `x` về 0 mọi bit đã được xét đúng một lần (điều kiện dừng `x = 0` an toàn vì khi `x = 0` không còn bit 1 nào).
- **Invariant:** trước mỗi vòng, `count + popcount(x) = popcount(i)`. Lúc đầu `count = 0`, `x = i` nên đúng. Mỗi vòng chuyển một bit từ `x` sang `count` (nếu nó là 1) nên đẳng thức giữ nguyên; khi `x = 0` thì `count = popcount(i)`.
- **Cộng đúng giá trị:** ta cộng `nums[i]` (giá trị tại đúng chỉ số được chọn), nên tổng là đáp án theo định nghĩa của đề.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`. Mỗi chỉ số `i < n` có tối đa `⌊log₂ n⌋ + 1` bit.

| Cách | Time | Space |
|---|---|---|
| 1. `bin().count` | `O(n log n)` | `O(log n)` |
| 2. Dịch bit | `O(n log n)` | `O(1)` |
| 3. Kernighan | `O(n log n)` (xấu nhất), ít hơn trung bình | `O(1)` |
| 4. Bảng DP | `O(n)` | `O(n)` |
| 5. `bit_count()` | `O(n)` (số nhỏ) | `O(1)` |

Với `n ≤ 1000`, `log₂ 1000 ≈ 10`, tức mỗi chỉ số tối đa 10 bit. Số thao tác chỉ cỡ `10 000`: mọi cách đều chạy trong chớp mắt. Sự khác biệt thật sự nằm ở **mức hiểu bit**, không ở tốc độ.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Gom mọi lời giải thành một danh sách để chạy cùng bộ test.

In [19]:
sols = [SolutionShift(), SolutionKernighan(), SolutionTable(), SolutionBitCount(), Solution()]

def brute(nums, k):
    return sum_k_bits_bruteforce(nums, k)

cases = [
    ("ví dụ 1", [5, 10, 1, 5, 2], 1, 13),
    ("ví dụ 2", [4, 3, 2, 1], 2, 1),
    ("1 phần tử, k=0 (chỉ số 0)", [7], 0, 7),
    ("1 phần tử, k=1", [7], 1, 0),
    ("k=0 chỉ chọn chỉ số 0", [9, 8, 7, 6, 5], 0, 9),
    ("k lớn hơn mọi popcount", [1, 2, 3], 10, 0),
    ("k=3 trên 8 phần tử (chỉ số 7)", [1, 2, 3, 4, 5, 6, 7, 8], 3, 8),
    ("toàn số 1, k=2, n=10", [1] * 10, 2, 4),   # chỉ số 3, 5, 6, 9
]
for name, nums, k, exp in cases:
    assert brute(nums, k) == exp, (name, "brute", brute(nums, k), exp)
    for s in sols:
        got = s.sumIndicesWithKSetBits(nums, k)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 13
PASS ví dụ 2 -> 1
PASS 1 phần tử, k=0 (chỉ số 0) -> 7
PASS 1 phần tử, k=1 -> 0
PASS k=0 chỉ chọn chỉ số 0 -> 9
PASS k lớn hơn mọi popcount -> 0
PASS k=3 trên 8 phần tử (chỉ số 7) -> 8
PASS toàn số 1, k=2, n=10 -> 4


### 12.2. Phản ví dụ cho lỗi hay gặp

Mảng mà **giá trị** có nhiều bit nhưng **chỉ số** thì ít bit: nếu lẫn giá trị với chỉ số code sẽ sai ngay.

In [20]:
nums = [7, 7, 7, 7]    # 7 = 111 có 3 bit 1, nhưng chỉ số 0..3 có tối đa 2 bit 1
assert Solution().sumIndicesWithKSetBits(nums, 3) == 0   # không chỉ số nào có 3 bit
assert Solution().sumIndicesWithKSetBits(nums, 2) == 7   # chỉ có chỉ số 3
assert Solution().sumIndicesWithKSetBits(nums, 1) == 14  # chỉ số 1 và 2
print("PASS phản ví dụ giá trị vs chỉ số")

PASS phản ví dụ giá trị vs chỉ số


### 12.3. Đối chiếu ngẫu nhiên (có seed) trên đúng miền của đề

Sinh `n ∈ [1, 1000]`, `nums[i] ∈ [1, 10^5]`, `k ∈ [0, 10]`, rồi so mọi cách với brute force.

In [21]:
import random
random.seed(2859)
for t in range(300):
    n = random.randint(1, 1000)
    nums = [random.randint(1, 10**5) for _ in range(n)]
    k = random.randint(0, 10)
    exp = brute(nums, k)
    for s in sols:
        assert s.sumIndicesWithKSetBits(nums, k) == exp, (t, type(s).__name__)
print("300 test ngẫu nhiên: mọi cách khớp brute force")

300 test ngẫu nhiên: mọi cách khớp brute force


In [22]:
# Kiểm cả bảng popcount: 4 cách đếm bit phải trùng nhau trên 0..2047
for i in range(2048):
    a = popcount_str(i)
    assert a == popcount_shift(i) == popcount_kernighan(i) == i.bit_count()
pc = [0] * 2048
for i in range(1, 2048):
    pc[i] = pc[i >> 1] + (i & 1)
assert all(pc[i] == popcount_str(i) for i in range(2048))
print("5 cách đếm bit trùng khớp trên 0..2047")

5 cách đếm bit trùng khớp trên 0..2047


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm bit của **giá trị** thay vì **chỉ số**

In [23]:
def sai_dem_gia_tri(nums, k):
    return sum(v for i, v in enumerate(nums) if bin(v).count("1") == k)   # sai: dùng v thay vì i

print("sai :", sai_dem_gia_tri([5, 10, 1, 5, 2], 1))                 # chọn theo bit của 5,10,1,5,2
print("đúng:", Solution().sumIndicesWithKSetBits([5, 10, 1, 5, 2], 1))   # 13

sai : 3
đúng: 13


**Vì sao sai:** đề nói "chỉ số có đúng `k` set bit", không phải giá trị. `bin(v)` phải được thay bằng `bin(i)`.

### Lỗi 2: Đánh chỉ số từ 1

In [24]:
def sai_chi_so_tu_1(nums, k):
    return sum(v for i, v in enumerate(nums, start=1) if bin(i).count("1") == k)   # sai: i bắt đầu từ 1

print("sai :", sai_chi_so_tu_1([4, 3, 2, 1], 2))                 # chọn i=3 -> nums[2]=2
print("đúng:", Solution().sumIndicesWithKSetBits([4, 3, 2, 1], 2))   # 1

sai : 2
đúng: 1


**Vì sao sai:** đề nói mảng **0-indexed**; chỉ số đầu tiên là `0`. Dùng `enumerate(nums)` mặc định đã bắt đầu từ 0, đừng thêm `start=1`.

### Lỗi 3: Bỏ qua `k = 0`

In [25]:
def sai_bo_k0(nums, k):
    total = 0
    for i in range(1, len(nums)):          # sai: bỏ chỉ số 0 vì nghĩ "0 không có bit"
        if bin(i).count("1") == k:
            total += nums[i]
    return total

print("sai :", sai_bo_k0([9, 8, 7, 6, 5], 0))                      # 0
print("đúng:", Solution().sumIndicesWithKSetBits([9, 8, 7, 6, 5], 0))   # 9

sai : 0
đúng: 9


**Vì sao sai:** `0` viết nhị phân là `0`, có **đúng 0** set bit, nên chỉ số `0` được chọn khi `k = 0`. `k` có thể bằng 0 theo constraints; phải duyệt từ chỉ số `0`.

### Lỗi 4: Vòng `while x > 0` với đếm sai bit

In [26]:
def sai_dem_bit_0(x):
    count = 0
    while x:
        count += 1 - (x & 1)      # sai: đếm bit 0 thay vì bit 1
        x >>= 1
    return count

print("sai :", sai_dem_bit_0(5))         # 5 = 101 -> 1 bit 0
print("đúng:", popcount_shift(5))        # 2 bit 1

sai : 1
đúng: 2


**Vì sao sai:** đề hỏi bit **1** (set bit). Đếm bit 0 còn phụ thuộc cách ta viết độ rộng nên càng không ổn định.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy cụm "**set bits**", "**số bit 1**", "**popcount**", "**Hamming weight**" → nghĩ `bin(x).count("1")` hoặc `x.bit_count()`.
- Thấy "chỉ số / vị trí có tính chất gì đó → cộng giá trị tại vị trí" → tách làm **bộ lọc theo chỉ số** + `sum`.
- `bin(x).count("1")` là cách ngắn nhất để **nộp nhanh**; Kernighan / DP là cách để **hiểu sâu** và để trả lời phỏng vấn.
- Với `n ≤ 1000` đừng tối ưu quá; ưu tiên code đúng, ngắn, dễ đọc.
- Một dòng "nộp nhanh": `return sum(v for i, v in enumerate(nums) if bin(i).count("1") == k)`.

## 15. Biến thể của bài

1. **Chỉ số có số bit 1 là số nguyên tố**: chọn `i` nếu `popcount(i)` thuộc `{2, 3, 5, 7}` (liên quan LeetCode 762 *Prime Number of Set Bits in Binary Representation*).
2. **Đếm số chỉ số** (thay vì cộng giá trị) có đúng `k` bit.
3. **Chỉ số có số bit 1 chẵn** ("evil number"), hoặc lẻ ("odious number").
4. **Đếm bit trên một đoạn `[L, R]`**: nhớ lại LeetCode 338 *Counting Bits*, trong đó bảng DP `pc[i] = pc[i >> 1] + (i & 1)` là lời giải kinh điển.
5. **Dùng `k` nhỏ để sinh nhanh các chỉ số** bằng tổ hợp: chọn `k` vị trí bit trong `m` bit.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `bin().count` | đổi sang chuỗi rồi đếm "1" | `O(n log n)` | `O(log n)` | nộp nhanh, dễ đọc nhất |
| 2. Dịch bit | `x & 1`, `x >>= 1` | `O(n log n)` | `O(1)` | học bit; không cần chuỗi |
| 3. Kernighan | `x &= x - 1` | `O(n log n)` (≤ số bit 1) | `O(1)` | số bit 1 ít; câu hỏi phỏng vấn |
| 4. Bảng DP | `pc[i] = pc[i>>1] + (i&1)` | `O(n)` | `O(n)` | cần popcount của cả đoạn liên tiếp |
| 5. `bit_count()` | hàm dựng sẵn | `O(n)` | `O(1)` | Python ≥ 3.10, ngắn nhất |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `bin(10)`, `bin(10).count("1")`, `10 & 1`, `10 >> 1`, `12 & 11`.

**Bài 2 (tính tay).** Với `nums = [3, 1, 4, 1, 5, 9, 2, 6]`, `k = 2`: liệt kê chỉ số `0..7` ở dạng nhị phân 3 bit, đánh dấu chỉ số có đúng 2 bit 1 và tính tổng.

**Bài 3 (điền TODO).** Hoàn thành hàm `popcount_todo(x)` bằng Kernighan.

In [27]:
def popcount_todo(x):
    count = 0
    # TODO 1: while x:
    # TODO 2:     x &= x - 1
    # TODO 3:     count += 1
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `sum_k_bits_table(nums, k)` bằng bảng DP `pc[i] = pc[i >> 1] + (i & 1)` mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_indices(n, k)`: đếm số chỉ số `i` trong `0..n-1` có đúng `k` bit 1. (b) Viết `sum_prime_bits(nums)`: cộng `nums[i]` nếu số bit 1 của `i` là số nguyên tố.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `bin(10) = '0b1010'`, có 2 bit 1; `10 & 1 = 0`; `10 >> 1 = 5`; `12 & 11 = 1100 & 1011 = 1000 = 8`. Bài 2: viết bảng 3 bit: `000, 001, 010, 011, 100, 101, 110, 111`.
- **Mức 2 (vừa):** Bài 3: ba dòng đã ghi sẵn trong comment. Bài 4: tạo `pc = [0] * n`, rồi `for i in range(1, n)`.
- **Mức 3 (sâu):** Bài 5(a): chỉ cần đếm, không cần mảng `nums`; dùng `sum(1 for i in range(n) if i.bit_count() == k)`. Bài 5(b): tự viết `is_prime` nhỏ, vì popcount ≤ 10 có thể dùng tập `{2, 3, 5, 7}`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert bin(10) == "0b1010" and bin(10).count("1") == 2
assert 10 & 1 == 0 and 10 >> 1 == 5 and 12 & 11 == 8

# Bài 2: nums = [3,1,4,1,5,9,2,6], k = 2 -> chỉ số 3 (011), 5 (101), 6 (110)
# nums[3] + nums[5] + nums[6] = 1 + 9 + 2 = 12
assert Solution().sumIndicesWithKSetBits([3, 1, 4, 1, 5, 9, 2, 6], 2) == 12

# Bài 3
def popcount_todo_ans(x):
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count

assert [popcount_todo_ans(i) for i in range(8)] == [0, 1, 1, 2, 1, 2, 2, 3]

# Bài 4
def sum_k_bits_table(nums, k):
    n = len(nums)
    pc = [0] * n
    for i in range(1, n):
        pc[i] = pc[i >> 1] + (i & 1)
    return sum(nums[i] for i in range(n) if pc[i] == k)

assert sum_k_bits_table([5, 10, 1, 5, 2], 1) == 13
assert sum_k_bits_table([4, 3, 2, 1], 2) == 1

# Bài 5(a)
def count_indices(n, k):
    return sum(1 for i in range(n) if bin(i).count("1") == k)

assert count_indices(8, 1) == 3        # 1, 2, 4
assert count_indices(8, 0) == 1        # chỉ số 0
assert count_indices(16, 2) == 6       # C(4,2)

# Bài 5(b)
def sum_prime_bits(nums):
    primes = {2, 3, 5, 7}
    return sum(v for i, v in enumerate(nums) if bin(i).count("1") in primes)

# 8 phần tử: popcount = [0,1,1,2,1,2,2,3] -> chỉ số 3,5,6 (2 bit) và 7 (3 bit)
assert sum_prime_bits([1, 1, 1, 1, 1, 1, 1, 1]) == 4
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lọc theo thuộc tính nhị phân của chỉ số)

```python
from typing import List

class Solution:
    def sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int:
        return sum(v for i, v in enumerate(nums) if bin(i).count("1") == k)
```

**Checklist**

- [ ] Đếm bit của **chỉ số** `i`, không phải của `nums[i]`?
- [ ] Mảng 0-indexed: `enumerate(nums)` không có `start=1`?
- [ ] `k = 0` vẫn chọn chỉ số `0`?
- [ ] Biết ít nhất hai cách đếm bit: `bin().count("1")` và `x & (x - 1)`?
- [ ] Đã thử `n = 1`, `k` lớn hơn mọi popcount, và `k = 0`?
- [ ] Signature đúng `sumIndicesWithKSetBits(self, nums: List[int], k: int) -> int`?